# 06 — Find the ZP worker's context char/token ceiling

Sister notebook to `05_debug_context_attribution`. There we sidestepped the
worker's ~2k-token ceiling by running ContextCite **once per chunk**. This
notebook asks the prior question directly:

> **How big can a single `prompt_logprobs=1` call get before the `local-model-mini`
> worker fails?** — and therefore, can we ever feed *all chunks at once* and do
> chunk attribution together rather than per-chunk?

**What this notebook does:**

- **No ContextCite, no ablations, no Lasso.** We bypass all of that and hit the
  module's call chokepoint `_api_response_token_logprobs(user_content, response)`
  directly — the exact call ContextCite makes internally.
- **Incremental sweep.** Concatenate the mock's chunks into one context, then
  replay the same prompt at a growing series of char budgets. For each budget we
  record: HTTP status, server-side prompt-token count, and pass/fail.
- **No backend API call.** The RAG response is loaded from the local
  `chat_output.txt` mock, same as 05.

Flow: `load mock → response → sweep growing context → find the cliff`.

## Parameters

All knobs for the run live here. Tweak and re-run top-to-bottom.

In [ ]:
# ============================ Parameters ============================

# Mock backend payload to debug against (no live backend call is made).
MOCK_PATH = 'chat_output.txt'
QUERY     = 'Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?'

# Response cap (chars). The response rides in every prompt_logprobs call, so it
# eats part of the budget too. Keep it fixed across the sweep so the only thing
# growing is the context.
MAX_RESPONSE_CHARS = 500

# --- The sweep ---
# Context char budgets to probe, smallest -> largest. Each budget builds a context
# of (up to) that many chars from the concatenated chunks and fires ONE call.
# Tune the range/step to bracket wherever the worker starts 500-ing.
SWEEP_START =  500
SWEEP_STOP  = 8000   # mock has ~7.7k chars of context total
SWEEP_STEP  =  500

# Stop sweeping after this many consecutive failures (the worker is likely wedged
# past the cliff; no point hammering it). Set to None to probe every budget.
STOP_AFTER_CONSEC_FAILS = 3

# --- Graceful calls: throttle + retry ---
# A failure here is the SIGNAL we're hunting, not noise to ride through — so keep
# retries low. The delay just spaces calls so a burst can't muddy the result.
API_CALL_DELAY    = 1.0   # seconds to wait BEFORE each call
API_MAX_RETRIES   = 1     # 1 = no retry; a 5xx is recorded as a fail, not retried
API_RETRY_BACKOFF = 3.0   # linear backoff (only matters if retries > 1)

# Logging: when True the module logs per-call token counts at INFO.
VERBOSE_API_LOGGING = True

## Setup — import the module and inspect its config

In [ ]:
import sys, json, logging
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Import the module under test. It self-configures from .env on import:
# the Zingplay LLM endpoint for prompt_logprobs.
import context_attribution.context_cite as cc_module
from context_attribution.context_cite import prepare_inputs, make_tokenizer

# Response cap from Parameters. We don't truncate the concatenated context via the
# module — the sweep controls the context size explicitly, budget by budget.
cc_module.MINI_MAX_RESPONSE_CHARS = MAX_RESPONSE_CHARS
cc_module.MINI_MAX_CONTEXT_CHARS  = None

# Apply graceful-call settings (throttle + retry) to the module's call chokepoint.
cc_module.API_CALL_DELAY    = API_CALL_DELAY
cc_module.API_MAX_RETRIES   = API_MAX_RETRIES
cc_module.API_RETRY_BACKOFF = API_RETRY_BACKOFF

# Logging: keep the root quiet but turn the module's per-call token logging up to
# INFO so the size of each probe surfaces in the notebook.
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s', force=True)
logging.getLogger('context_attribution.context_cite').setLevel(
    logging.INFO if VERBOSE_API_LOGGING else logging.WARNING
)

print('Module under test   :', cc_module.__file__)
print('LLM endpoint (ZP)   :', cc_module.LLM_URL)
print('Model               :', cc_module.MODEL)
print('LLM API key set     :', bool(cc_module.API_KEY))
print('Response cap (chars):', cc_module.MINI_MAX_RESPONSE_CHARS)
print(f'Throttle / retry    : delay={API_CALL_DELAY}s, retries={API_MAX_RETRIES}, backoff={API_RETRY_BACKOFF}s')
print('Verbose API logging :', VERBOSE_API_LOGGING)

# Sanity: we need the Zingplay endpoint, but NOT the backend.
assert cc_module.API_KEY,  'OPENAI_API_KEY missing — needs the Zingplay endpoint'
assert cc_module.LLM_URL.startswith('http'), f'LLM_URL looks unset: {cc_module.LLM_URL!r}'
print('\nReady: Zingplay endpoint configured, backend will NOT be called.')

## Health check — is the model backend alive?

A trivial `"hello"` call (`max_tokens=1`) against the Zingplay endpoint, run
*before* any heavy calls. If this 500s, the `local-model-mini` worker is down /
unreachable (e.g. crashed by an earlier oversized prompt) — fix the backend
before sweeping; the sweep results would be meaningless. This mirrors
`context_attribution/tests/test_zp_api.py`.

In [ ]:
import requests

r = requests.post(
    cc_module.LLM_URL,
    headers={'Authorization': f'Bearer {cc_module.API_KEY}'},
    json={
        'model': cc_module.MODEL,
        'messages': [{'role': 'user', 'content': 'hello'}],
        'max_tokens': 1,
        'temperature': 0,
    },
    timeout=30,
)

print(f'POST {cc_module.LLM_URL}')
print(f'model       : {cc_module.MODEL}')
print(f'status      : {r.status_code}')
print(f'body (head) : {r.text[:200]}')

if r.ok:
    print('\nBackend is ALIVE — safe to proceed.')
else:
    print('\n*** Backend is DOWN / unreachable. Fix the worker before sweeping. ***')

assert r.ok, f'Model backend not healthy (HTTP {r.status_code}) — fix the endpoint before continuing.'

## Stage 1 — Load mock data (no backend call)

`chat_output.txt` is a captured backend payload (the same shape `fetch_backend`
returns). We load it straight from disk and pair it with the query that produced
it, so the backend is bypassed entirely.

In [ ]:
# MOCK_PATH and QUERY come from the Parameters cell.
with open(MOCK_PATH) as f:
    data = json.load(f)

print('Loaded mock payload :', Path(MOCK_PATH).resolve())
print('Query               :', QUERY)
print(f'answer              : {len(data["answer"])} chars')
print(f'knowledge_sources   : {len(data["knowledge_sources"])} chunks')
print()
for i, src in enumerate(data['knowledge_sources']):
    print(f'  [{i}] {len(src.get("content", "")):>5} chars  score={src.get("score", 0):.3f}  {src.get("document_id", "")[:50]}')

## Stage 2 — Build the full concatenated context + capped response

`prepare_inputs` applies the response cap and concatenates **all** chunks into one
context (we set `MINI_MAX_CONTEXT_CHARS = None` in Setup, so nothing is dropped).
This concatenated context is exactly what we'd feed if we wanted to attribute every
chunk together — the sweep below grows a prefix of it until the worker chokes.

In [ ]:
context, response = prepare_inputs(data)

print(f'\nFull concatenated context : {len(context)} chars')
print(f'Capped response           : {len(response)} chars')
print('\n--- response ---')
print(response)

## Stage 3 — Incremental sweep: grow the context until the worker fails

For each char budget in the sweep we:
1. Take a `budget`-char prefix of the full concatenated context.
2. Wrap it in the same ChatML user-content the attribution path uses
   (`prompt_template.format(context=..., query=...)`).
3. Fire ONE `_api_response_token_logprobs(user_content, response)` call — the exact
   chokepoint ContextCite hits internally.
4. Record status (`ok` / HTTP code / exception), server-side prompt-token count,
   and a GPT-2 estimate of the prompt size.

The first budget that fails — and the prompt-token count just below it — is the
worker's ceiling. If even the full ~7.7k-char context passes, we can attribute all
chunks in one run and the per-chunk split in 05 is unnecessary.

In [ ]:
import requests as _requests
import pandas as pd

# Same prompt template ContextCite uses, so the probe matches the real call shape.
PROMPT_TEMPLATE = (
    'Context: {context}\n\nQuery: {query}'
)


def probe(budget_chars):
    """Fire one prompt_logprobs call at the given context-char budget.
    Returns a result dict; never raises (failures are the signal we want)."""
    ctx = context[:budget_chars]
    user_content = PROMPT_TEMPLATE.format(context=ctx, query=QUERY)
    est_tokens = (cc_module._estimate_tokens(user_content)
                  + cc_module._estimate_tokens(response))
    row = {
        'budget_chars':   budget_chars,
        'ctx_chars':      len(ctx),
        'user_chars':     len(user_content),
        'est_tokens_gpt2': est_tokens,
        'status':         None,
        'server_tokens':  None,
        'ok':             False,
        'detail':         '',
    }
    try:
        toks = cc_module._api_response_token_logprobs(user_content, response)
        row['ok'] = True
        row['status'] = 200
        row['server_tokens'] = len(toks)  # response tokens scored back
        row['detail'] = f'{len(toks)} response tokens scored'
    except _requests.HTTPError as e:
        resp = e.response
        row['status'] = resp.status_code if resp is not None else 'HTTPError'
        row['detail'] = (resp.text[:160] if resp is not None else str(e))
    except Exception as e:
        row['status'] = e.__class__.__name__
        row['detail'] = str(e)[:160]
    return row


budgets = list(range(SWEEP_START, SWEEP_STOP + 1, SWEEP_STEP))
results, consec_fail = [], 0

for b in budgets:
    row = probe(b)
    results.append(row)
    flag = 'OK ' if row['ok'] else 'FAIL'
    print(f"[{flag}] budget={b:>5}  ctx={row['ctx_chars']:>5}  "
          f"user={row['user_chars']:>5} chars  ~{row['est_tokens_gpt2']:>4} gpt2-tok  "
          f"status={row['status']}  {row['detail'][:60]}")
    consec_fail = 0 if row['ok'] else consec_fail + 1
    if STOP_AFTER_CONSEC_FAILS and consec_fail >= STOP_AFTER_CONSEC_FAILS:
        print(f'\nStopping: {consec_fail} consecutive failures — past the cliff.')
        break

sweep_df = pd.DataFrame(results)
print('\n--- sweep results ---')
sweep_df

## Stage 4 — The verdict: where is the cliff?

Largest budget that passed vs. smallest that failed. This is the practical ceiling
for a single all-chunks call. Compare the largest passing `ctx_chars` against the
full context size to decide: **one combined run, or per-chunk (05)?**

In [ ]:
ok_rows   = sweep_df[sweep_df['ok']]
fail_rows = sweep_df[~sweep_df['ok']]

print(f'Full concatenated context : {len(context)} chars\n')

if not ok_rows.empty:
    best = ok_rows.iloc[-1]
    print(f'Largest PASSING : budget={best.budget_chars}  '
          f'ctx={best.ctx_chars} chars  ~{best.est_tokens_gpt2} gpt2-tok')
else:
    print('No budget passed — even the smallest probe failed.')

if not fail_rows.empty:
    worst = fail_rows.iloc[0]
    print(f'Smallest FAILING: budget={worst.budget_chars}  '
          f'ctx={worst.ctx_chars} chars  ~{worst.est_tokens_gpt2} gpt2-tok  '
          f'(status {worst.status})')
else:
    print('No budget failed in the swept range.')

print()
if not ok_rows.empty and ok_rows.iloc[-1].ctx_chars >= len(context):
    print('VERDICT: the full context fits in one call — you CAN attribute all '
          'chunks together. The per-chunk split in 05 is not strictly required.')
elif not ok_rows.empty:
    print(f'VERDICT: the worker caps out around {ok_rows.iloc[-1].ctx_chars} ctx '
          f'chars (< full {len(context)}). All-chunks-at-once will NOT fit — '
          'per-chunk (05) remains necessary.')
else:
    print('VERDICT: inconclusive — widen / lower the sweep range and re-run.')